In [10]:
from tqdm import tqdm
from torchinfo import summary
import matplotlib.pyplot as plt
import data_proc.data_preproc as data_preproc
import cutouts.cutouts as cutouts
import numpy as np
import torchvision.transforms as T
from sklearn.linear_model import LogisticRegression
import torch.nn as nn
import torch
from sklearn.neighbors import KNeighborsClassifier
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import umap.umap_ as umap

In [11]:
import simclr.simclr as simclr

In [12]:
eddy_loader, eddy_loader_val = data_preproc.get_dataloaders(data = "eddy", size=256)

In [13]:

def knn_score(X, y, X_val, y_val, k=5):
    neigh = KNeighborsClassifier(n_neighbors=k)
    neigh.fit(X, y)

    print(neigh.score(X_val, y_val))
    
def linear_probe_score(X, y, X_val, y_val, max_iter=2000, C=1.0):
    """
    Linear probe using multinomial logistic regression.

    X, X_val: [N, D] feature arrays
    y, y_val: [N] labels
    """
    
    mu = X.mean(axis=0)
    sigma = X.std(axis=0) + 1e-6

    Xn = (X - mu) / sigma
    Xvn = (X_val - mu) / sigma

    clf = LogisticRegression(
        max_iter=max_iter,
        C=C,
        solver="lbfgs",
        verbose=2
    )
    clf.fit(Xn, y)

    acc = clf.score(Xvn, y_val)
    print("linear probe accuracy:", acc)
    return acc, clf

In [37]:
def train_one_epoch(
    model: nn.Module,
    dataloader,
    transforms,
    optimizer: torch.optim.Optimizer,
    loss_fn,
    device: str = "cuda",
    image_key: str = "image",
    resnet= False
):

    model.train()

    # model.backbone.eval()

    running_loss = 0.0
    num_batches = 0

    for batch in tqdm(dataloader, total=len(dataloader)):
        if isinstance(batch, dict):
            x = batch[image_key]
        else:
            x = batch[0]

        xi = transforms(x)
        xj = transforms(x)

        xi = xi.to(device, non_blocking=True)
        xj = xj.to(device, non_blocking=True)

        if (resnet):
            zi = model(xi)
            zj = model(xj)
        else:
            _, zi = model(xi)
            _, zj = model(xj)

        loss = loss_fn(zi, zj)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()


        # num_with_grad = 0
        # for name, p in model.backbone.named_parameters():
        #     if p.grad is not None:
        #         g = p.grad.abs().mean().item()
        #         if g > 0:
        #             print(name, g)
        #             num_with_grad += 1
        # print("num backbone params with nonzero grad:", num_with_grad)

        

        optimizer.step()

        running_loss += loss.item()
        num_batches += 1

    return running_loss / max(num_batches, 1)

In [15]:
def frozen_features(loader, model):
    all_feats = []
    all_labels = []
    
    for batch in tqdm(loader, total=len(loader)):
        with torch.no_grad():
            proj = model.input_projector(batch["image"].to("cuda"))
            features = model.backbone(proj)
            features = features / features.norm(dim=-1, keepdim=True)

            all_feats.append(features.cpu())
            all_labels.append(batch["label"].cpu())

    X = torch.cat(all_feats, dim=0)
    y = torch.cat(all_labels, dim=0)

    return X, y

def check_knn_accuracy(model, loader, loader_val):
    X, y  = frozen_features(loader, model)
    X_val, y_val  = frozen_features(loader_val, model)
    
    knn_score(X, y, X_val, y_val, k=5)
    linear_probe_score(X, y, X_val, y_val, max_iter=2000, C=1.0)

In [43]:
def fit(
    model: nn.Module,
    train_loader,
    loader_val,
    optimizer: torch.optim.Optimizer,
    loss_fn,
    epochs: int,
    device: str = "cuda",
    transforms=None,
    image_key: str = "image",
    resnet = False
):
    model.to(device)

    for epoch in range(epochs):
        avg_loss = train_one_epoch(
            model=model,
            dataloader=train_loader,
            transforms=transforms,
            optimizer=optimizer,
            loss_fn=loss_fn,
            device=device,
            image_key=image_key,
            resnet=resnet
        )

        
        print(f"Epoch {epoch+1:03d} | loss={avg_loss:.6f}")

        if (epoch % 10 == 0):
            if (resnet):
                check_knn_accuracy_resnet(model, train_loader, loader_val)
            else:
                check_knn_accuracy(model, train_loader, loader_val)

In [28]:
dino_model = torch.hub.load('facebookresearch/dino:main', 'dino_vits8')
backbone_feat_dim = 384

Using cache found in /home/jovyan/.cache/torch/hub/facebookresearch_dino_main


In [29]:
simclr_transforms_strong = T.Compose([
        T.RandomResizedCrop(64, scale=(0.3, 1.0)),
        T.RandomHorizontalFlip(),
        # transforms.RandomApply([transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
        T.RandomGrayscale(p=0.2),
        T.GaussianBlur(kernel_size=3),
        # transforms.ToTensor()
    ])

simclr_transforms_weak = T.Compose([
        T.RandomResizedCrop(64, scale=(0.8, 1.0)),
        T.RandomHorizontalFlip(),
        # transforms.RandomApply([transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
        # T.RandomGrayscale(p=0.2),
        T.GaussianBlur(kernel_size=3, sigma=(0.1, 0.5)),
        # transforms.ToTensor()
    ])

# Frozen Dino backbone

In [12]:
model = simclr.FrozenDinoSimCLR(
    dino_backbone=dino_model,
    in_channels=3,
    backbone_feat_dim=backbone_feat_dim,
    proj_hidden_dim=2048,
    proj_out_dim=128,
    freeze_backbone=True,
)

# Only optimize trainable modules
optimizer = torch.optim.AdamW(
    list(model.input_projector.parameters()) + list(model.projection_head.parameters()),
    lr=1e-3,
    weight_decay=1e-4,
)

transforms = simclr_transforms_strong

fit(
    model=model,
    train_loader=eddy_loader,
    loader_val=eddy_loader_val,
    optimizer=optimizer,
    loss_fn=simclr.nt_xent_loss,
    epochs=100,
    device="cuda",
    transforms=transforms,
    image_key="image",
)

100%|██████████| 61/61 [00:31<00:00,  1.93it/s]


Epoch 001 | loss=3.913813


100%|██████████| 15/15 [00:06<00:00,  2.39it/s]


0.6242105263157894
linear probe accuracy: 0.6294736842105263


100%|██████████| 61/61 [00:31<00:00,  1.95it/s]


...

Epoch 041 | loss=3.320501


100%|██████████| 15/15 [00:06<00:00,  2.41it/s]


0.5821052631578948
linear probe accuracy: 0.6326315789473684


 51%|█████     | 31/61 [00:16<00:15,  1.93it/s]


KeyboardInterrupt: 

# Unfrozen backbone with initialization

In [13]:
# not frozen dino
model = simclr.FrozenDinoSimCLR(
    dino_backbone=dino_model,
    in_channels=3,
    backbone_feat_dim=backbone_feat_dim,
    proj_hidden_dim=2048,
    proj_out_dim=128,
    freeze_backbone=False,
)

# Only optimize trainable modules
optimizer = torch.optim.AdamW(
    list(model.parameters()),
    lr=1e-3,
    weight_decay=1e-4,
)

transforms = simclr_transforms_strong

fit(
    model=model,
    train_loader=eddy_loader,
    loader_val=eddy_loader_val,
    optimizer=optimizer,
    loss_fn=simclr.nt_xent_loss,
    epochs=100,
    device="cuda",
    transforms=transforms,
    image_key="image",
)

100%|██████████| 61/61 [00:31<00:00,  1.97it/s]


Epoch 001 | loss=3.929295


100%|██████████| 15/15 [00:05<00:00,  2.52it/s]


0.68
linear probe accuracy: 0.6747368421052632


100%|██████████| 61/61 [00:31<00:00,  1.96it/s]


...

Epoch 052 | loss=3.305065


100%|██████████| 61/61 [00:31<00:00,  1.95it/s]


Epoch 053 | loss=3.327267


 62%|██████▏   | 38/61 [00:19<00:11,  1.95it/s]


KeyboardInterrupt: 

# Uninitialized backbone not frozen

In [14]:
# No init on dino 
dino_model = torch.hub.load('facebookresearch/dino:main', 'dino_vits8', pretrained=False,)

model = simclr.FrozenDinoSimCLR(
    dino_backbone=dino_model,
    in_channels=3,
    backbone_feat_dim=backbone_feat_dim,
    proj_hidden_dim=2048,
    proj_out_dim=128,
    freeze_backbone=False,
)

optimizer = torch.optim.AdamW(
    list(model.parameters()),
    lr=1e-3,
    weight_decay=1e-4,
)

transforms = simclr_transforms_strong

fit(
    model=model,
    train_loader=eddy_loader,
    loader_val=eddy_loader_val,
    optimizer=optimizer,
    loss_fn=simclr.nt_xent_loss,
    epochs=100,
    device="cuda",
    transforms=transforms,
    image_key="image",
)

Using cache found in /home/jovyan/.cache/torch/hub/facebookresearch_dino_main
100%|██████████| 61/61 [00:33<00:00,  1.81it/s]


Epoch 001 | loss=3.949191


100%|██████████| 15/15 [00:05<00:00,  2.51it/s]


0.6610526315789473
linear probe accuracy: 0.6463157894736842


100%|██████████| 61/61 [00:33<00:00,  1.82it/s]


...

0.6557894736842105
linear probe accuracy: 0.6347368421052632


100%|██████████| 61/61 [00:33<00:00,  1.81it/s]


Epoch 012 | loss=3.494940


 31%|███       | 19/61 [00:10<00:23,  1.79it/s]


KeyboardInterrupt: 

# Looking at transforms

In [18]:
class AddGaussianNoise:
    def __init__(self, std=0.01):
        self.std = std

    def __call__(self, x):
        return x + torch.randn_like(x) * self.std

class RandomChannelDropout:
    def __init__(self, p=0.2):
        self.p = p

    def __call__(self, x):
        if torch.rand(1).item() < self.p:
            c = torch.randint(0, x.shape[0], (1,)).item()
            x = x.clone()
            x[c] = 0
        return x

In [24]:
simclr_transforms_physical = T.Compose([
        T.RandomResizedCrop(64, scale=(0.6, 1.0)),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomVerticalFlip(p=0.5),
        AddGaussianNoise(std=0.01),
        T.GaussianBlur(kernel_size=3, sigma=(0.1, 0.5)),
        RandomChannelDropout(p=0.1),
    ])

simclr_transforms_nonphysical = T.Compose([
        T.RandomResizedCrop(64, scale=(0.5, 1.0)),
        T.RandomHorizontalFlip(),
        T.RandomApply([T.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
        T.RandomGrayscale(p=0.2),
        T.GaussianBlur(kernel_size=3),
    ])


In [33]:
# Phyisical transforms

# No init on dino 
dino_model = torch.hub.load('facebookresearch/dino:main', 'dino_vits8', pretrained=False,)
backbone_feat_dim = 384

model = simclr.FrozenDinoSimCLR(
    dino_backbone=dino_model,
    in_channels=3,
    backbone_feat_dim=backbone_feat_dim,
    proj_hidden_dim=2048,
    proj_out_dim=128,
    freeze_backbone=False,
)

# Only optimize trainable modules
optimizer = torch.optim.AdamW(
    list(model.parameters()),
    lr=1e-3,
    weight_decay=1e-4,
)

transforms = simclr_transforms_physical

fit(
    model=model,
    train_loader=eddy_loader,
    loader_val=eddy_loader_val,
    optimizer=optimizer,
    loss_fn=simclr.nt_xent_loss,
    epochs=100,
    device="cuda",
    transforms=transforms,
    image_key="image",
)

Using cache found in /home/jovyan/.cache/torch/hub/facebookresearch_dino_main
100%|██████████| 61/61 [00:56<00:00,  1.09it/s]


Epoch 001 | loss=3.743332


100%|██████████| 15/15 [00:21<00:00,  1.40s/it]


0.47578947368421054
linear probe accuracy: 0.4936842105263158


100%|██████████| 61/61 [00:54<00:00,  1.11it/s]


...

0.54
linear probe accuracy: 0.5189473684210526


100%|██████████| 61/61 [00:55<00:00,  1.11it/s]


Epoch 022 | loss=3.259128


 54%|█████▍    | 33/61 [00:30<00:25,  1.09it/s]


KeyboardInterrupt: 

In [32]:
# non Phyisical transforms

# No init on dino 
dino_model = torch.hub.load('facebookresearch/dino:main', 'dino_vits8', pretrained=False,)
backbone_feat_dim = 384

model = simclr.FrozenDinoSimCLR(
    dino_backbone=dino_model,
    in_channels=3,
    backbone_feat_dim=backbone_feat_dim,
    proj_hidden_dim=2048,
    proj_out_dim=128,
    freeze_backbone=False,
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)

transforms = simclr_transforms_nonphysical

fit(
    model=model,
    train_loader=eddy_loader,
    loader_val=eddy_loader_val,
    optimizer=optimizer,
    loss_fn=simclr.nt_xent_loss,
    epochs=100,
    device="cuda",
    transforms=transforms,
    image_key="image",
)

Using cache found in /home/jovyan/.cache/torch/hub/facebookresearch_dino_main
100%|██████████| 61/61 [00:52<00:00,  1.17it/s]


Epoch 001 | loss=4.515162


100%|██████████| 15/15 [00:20<00:00,  1.37s/it]


0.6305263157894737
linear probe accuracy: 0.5831578947368421


100%|██████████| 61/61 [00:55<00:00,  1.10it/s]


...

Epoch 006 | loss=4.376318


100%|██████████| 61/61 [00:55<00:00,  1.10it/s]


Epoch 007 | loss=4.233743


  3%|▎         | 2/61 [00:01<00:53,  1.11it/s]


KeyboardInterrupt: 

# resnet simclr

In [44]:
def frozen_features_resnet(loader, model):
    all_feats = []
    all_labels = []

    model.eval()
    for batch in tqdm(loader, total=len(loader)):
        with torch.no_grad():
            features = model.encoder(batch["image"].to("cuda"))
            features = features / features.norm(dim=-1, keepdim=True)

            all_feats.append(features.cpu())
            all_labels.append(batch["label"].cpu())

    X = torch.cat(all_feats, dim=0)
    y = torch.cat(all_labels, dim=0)

    return X, y

def check_knn_accuracy_resnet(model, loader, loader_val):
    X, y  = frozen_features_resnet(loader, model)
    X_val, y_val  = frozen_features_resnet(loader_val, model)
    
    knn_score(X, y, X_val, y_val, k=5)
    linear_probe_score(X, y, X_val, y_val, max_iter=2000, C=1.0)

In [45]:
import torchvision.models as models

class SimCLRModel(nn.Module):
    def __init__(self, projection_dim=128):
        super().__init__()
        base_model = models.resnet18(weights=None)
        num_ftrs = base_model.fc.in_features
        base_model.fc = nn.Identity()
        self.encoder = base_model
        self.projection_head = nn.Sequential(
            nn.Linear(num_ftrs, 512),
            nn.ReLU(),
            nn.Linear(512, projection_dim)
        )

    def forward(self, x):
        h = self.encoder(x)
        z = self.projection_head(h)
        return z

In [46]:
model = SimCLRModel().to("cuda")

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)

transforms = simclr_transforms_physical

fit(
    model=model,
    train_loader=eddy_loader,
    loader_val=eddy_loader_val,
    optimizer=optimizer,
    loss_fn=simclr.nt_xent_loss,
    epochs=100,
    device="cuda",
    transforms=transforms,
    image_key="image",
    resnet=True
)

100%|██████████| 61/61 [00:49<00:00,  1.24it/s]


Epoch 001 | loss=3.357205


100%|██████████| 15/15 [00:09<00:00,  1.52it/s]


0.6452631578947369
linear probe accuracy: 0.716842105263158


100%|██████████| 61/61 [00:47<00:00,  1.29it/s]


...

Epoch 098 | loss=2.956318


100%|██████████| 61/61 [00:46<00:00,  1.32it/s]


Epoch 099 | loss=2.952425


100%|██████████| 61/61 [00:46<00:00,  1.31it/s]

Epoch 100 | loss=2.947855


# model from scratch

In [ ]:
simclr_transforms_physical = T.Compose([
        T.RandomResizedCrop(64, scale=(0.3, 1.0)),
        T.RandomHorizontalFlip(),
        # transforms.RandomApply([transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
        T.RandomGrayscale(p=0.2),
        T.GaussianBlur(kernel_size=3),
        # transforms.ToTensor()
    ])

In [ ]:
class PatchEmbed(nn.Module):
    def __init__(self, in_chans=8, img_size=64, patch_size=8, embed_dim=384):
        super().__init__()
        assert img_size % patch_size == 0, "img_size must be divisible by patch_size"
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2

        self.proj = nn.Conv2d(
            in_chans,
            embed_dim,
            kernel_size=patch_size,
            stride=patch_size,
        )

    def forward(self, x):
        # x: [B, C, H, W]
        x = self.proj(x)                  # [B, D, H/P, W/P]
        x = x.flatten(2).transpose(1, 2) # [B, N, D]
        return x


class MLPHead(nn.Module):
    def __init__(self, in_dim, hidden_dim=512, out_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x):
        return self.net(x)


class SimCLRViT(nn.Module):
    """
    8-channel ViT encoder + SimCLR projection head.

    Returns:
        h: encoder feature before projection head
        z: normalized projection for contrastive loss
    """
    def __init__(
        self,
        img_size=64,
        in_chans=8,
        patch_size=8,
        embed_dim=384,
        depth=6,
        num_heads=6,
        mlp_ratio=4.0,
        proj_hidden_dim=512,
        proj_out_dim=128,
        dropout=0.0,
    ):
        super().__init__()

        self.patch_embed = PatchEmbed(
            in_chans=in_chans,
            img_size=img_size,
            patch_size=patch_size,
            embed_dim=embed_dim,
        )
        num_patches = self.patch_embed.num_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, 1 + num_patches, embed_dim))
        self.pos_drop = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=int(embed_dim * mlp_ratio),
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=depth)
        self.norm = nn.LayerNorm(embed_dim)

        self.projection_head = MLPHead(
            in_dim=embed_dim,
            hidden_dim=proj_hidden_dim,
            out_dim=proj_out_dim,
        )

        self._init_weights()

    def _init_weights(self):
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def encode(self, x):
        # x: [B, C, H, W]
        x = self.patch_embed(x)  # [B, N, D]

        B = x.shape[0]
        cls = self.cls_token.expand(B, -1, -1)  # [B, 1, D]
        x = torch.cat([cls, x], dim=1)          # [B, 1+N, D]

        x = x + self.pos_embed[:, : x.size(1)]
        x = self.pos_drop(x)

        x = self.encoder(x)
        x = self.norm(x)

        h = x[:, 0]  # CLS token
        return h

    def forward(self, x):
        h = self.encode(x)                 # [B, embed_dim]
        z = self.projection_head(h)        # [B, proj_out_dim]
        z = F.normalize(z, dim=1)
        return h, z

In [ ]:
model = SimCLRViT(
    img_size=64,
    in_chans=8,
    patch_size=8,
    embed_dim=384,   # ViT-S style width
    depth=6,         # lighter than full ViT-S; increase to 12 if you want
    num_heads=6,
    proj_hidden_dim=512,
    proj_out_dim=128,
).cuda()

# VIC REG

In [47]:
import torch.nn.functional as F

def vicreg_loss(z1, z2, sim_coeff=25, std_coeff=25, cov_coeff=1):
    # invariance
    sim_loss = ((z1 - z2)**2).mean()

    # variance
    std_z1 = torch.sqrt(z1.var(dim=0) + 1e-4)
    std_z2 = torch.sqrt(z2.var(dim=0) + 1e-4)

    std_loss = torch.mean(F.relu(1 - std_z1)) + \
               torch.mean(F.relu(1 - std_z2))

    # covariance
    z1 = z1 - z1.mean(dim=0)
    z2 = z2 - z2.mean(dim=0)

    cov_z1 = (z1.T @ z1) / (z1.size(0) - 1)
    cov_z2 = (z2.T @ z2) / (z2.size(0) - 1)

    off_diag = lambda x: x.flatten()[1:].view(x.size(0)-1, x.size(1)+1)[:, :-1]

    cov_loss = off_diag(cov_z1).pow(2).mean() + \
               off_diag(cov_z2).pow(2).mean()

    return sim_coeff * sim_loss + std_coeff * std_loss + cov_coeff * cov_loss

In [48]:
#resnet with vicreg
model = SimCLRModel().to("cuda")

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)

transforms = simclr_transforms_physical

fit(
    model=model,
    train_loader=eddy_loader,
    loader_val=eddy_loader_val,
    optimizer=optimizer,
    loss_fn=vicreg_loss,
    epochs=100,
    device="cuda",
    transforms=transforms,
    image_key="image",
    resnet=True
)

100%|██████████| 61/61 [00:48<00:00,  1.24it/s]


Epoch 001 | loss=6.283989


100%|██████████| 15/15 [00:09<00:00,  1.50it/s]


0.6905263157894737
linear probe accuracy: 0.5989473684210527


100%|██████████| 61/61 [00:47<00:00,  1.28it/s]


...

Epoch 098 | loss=0.656378


100%|██████████| 61/61 [00:45<00:00,  1.34it/s]


Epoch 099 | loss=0.613082


100%|██████████| 61/61 [00:48<00:00,  1.27it/s]

Epoch 100 | loss=0.820665


In [18]:
dino_model = torch.hub.load('facebookresearch/dino:main', 'dino_vits8', pretrained=False,)

model = simclr.FrozenDinoSimCLR(
    dino_backbone=dino_model,
    in_channels=3,
    backbone_feat_dim=backbone_feat_dim,
    proj_hidden_dim=2048,
    proj_out_dim=128,
    freeze_backbone=False,
)

# Only optimize trainable modules
optimizer = torch.optim.AdamW(
    list(model.input_projector.parameters()) + list(model.projection_head.parameters()),
    lr=1e-3,
    weight_decay=1e-4,
)

transforms = simclr_transforms_strong

fit(
    model=model,
    train_loader=eddy_loader,
    loader_val=eddy_loader_val,
    optimizer=optimizer,
    loss_fn=vicreg_loss,
    epochs=100,
    device="cuda",
    transforms=transforms,
    image_key="image",
)

Using cache found in /home/jovyan/.cache/torch/hub/facebookresearch_dino_main
100%|██████████| 61/61 [00:33<00:00,  1.81it/s]


Epoch 001 | loss=45.848845


100%|██████████| 15/15 [00:06<00:00,  2.49it/s]


0.6578947368421053
linear probe accuracy: 0.6526315789473685


100%|██████████| 61/61 [00:33<00:00,  1.84it/s]


...

Epoch 009 | loss=45.674669


100%|██████████| 61/61 [00:33<00:00,  1.84it/s]


Epoch 010 | loss=45.663971


  7%|▋         | 4/61 [00:02<00:37,  1.52it/s]


KeyboardInterrupt: 